In [1]:
import pandas as pd
from sqlalchemy import create_engine

# Database connection details
db_user = 'root'
db_password = ''
db_host = 'localhost'
db_name = 'stock'
table_name = 'price'
backup_table_name = 'price_backup'

# Create a connection to the database
engine = create_engine("mysql+pymysql://root:@localhost:3306/stock")
const = engine.connect()

In [7]:
# Read the table into a Pandas DataFrame
df = pd.read_sql(f'SELECT * FROM price', con=const)
df.shape

(399793, 7)

In [4]:
# Back up the table
df.to_sql(backup_table_name, con=const, if_exists='replace', index=False)

In [5]:
# Read the table into a Pandas DataFrame
df_backup = pd.read_sql(f'SELECT * FROM price_backup', con=const)
df_backup.shape

(399793, 7)

In [9]:
# Drop duplicates based on 'name' and 'date' columns, keeping the last occurrence
df = df.drop_duplicates(subset=['name', 'date'], keep='last')

df.shape

(399301, 7)

In [10]:
# Write the updated DataFrame back to the MySQL table
df.to_sql(table_name, con=const, if_exists='replace', index=False)

print("Backup completed and duplicates deleted successfully.")

Backup completed and duplicates deleted successfully.


In [11]:
# Read the table into a Pandas DataFrame
df = pd.read_sql(f'SELECT * FROM price', con=const)
df.shape

(399301, 7)

In [3]:
import pandas as pd
from sqlalchemy import create_engine, text, exc
import traceback

try:
    # Create database connection
    engine = create_engine("mysql+pymysql://root:@localhost:3306/stock")
    
    # Test connection
    with engine.connect() as conn:
        conn.execute(text("SELECT 1"))
    print("✓ Database connection successful")
    
    # Read data
    df = pd.read_sql(text("SELECT * FROM price WHERE date = '2026-01-27'"), engine)
    print(f"✓ Read {len(df)} records for date 2026-01-27")
    
    if len(df) == 0:
        print("⚠ No records found for the specified date")
    else:
        # Check and remove duplicates
        duplicate_count = df.duplicated().sum()
        print(f"✓ Found {duplicate_count} duplicate records")
        
        df_clean = df.drop_duplicates()
        
        # Delete and re-insert
        with engine.begin() as connection:
            delete_result = connection.execute(
                text("DELETE FROM price WHERE date = '2026-01-27'")
            )
            print(f"✓ Deleted {delete_result.rowcount} records")
            
            df_clean.to_sql('price', connection, if_exists='append', index=False)
            print(f"✓ Inserted {len(df_clean)} clean records")
            
        print(f"✅ Success! Removed {duplicate_count} duplicates")
        
except exc.SQLAlchemyError as e:
    print(f"❌ Database error: {e}")
    traceback.print_exc()
except Exception as e:
    print(f"❌ Unexpected error: {e}")
    traceback.print_exc()

✓ Database connection successful
✓ Read 330 records for date 2026-01-27
✓ Found 165 duplicate records
✓ Deleted 330 records
✓ Inserted 165 clean records
✅ Success! Removed 165 duplicates
